# Quantum-circuit preprocessing for RL-based SWAP insertion

This notebook builds a **two-qubit backbone** for a routing agent:

1. Remove unitary one-qubit `Gate` objects and store them with their logical qubit and insertion slot.
2. Fuse consecutive unitary two-qubit gates when they act on the same pair of qubits and there is no instruction between them.
3. Keep barriers, measurements, resets, delays, and gates on 3+ qubits unchanged.

Adjacency is checked in the original circuit, not after deleting one-qubit gates. This preserves enough ordering information to restore the stored gates. A fused gate is represented by `UnitaryGate`, so a later basis-translation pass may decompose it into native hardware gates.

> **Routing note:** without SWAP insertion, `restore_without_routing` reconstructs a circuit equivalent to the input. After SWAP insertion, restore each stored gate at its recorded `slot`, translated through the router's logical-to-physical mapping at that slot. A helper for that translation is included below.

In [ ]:
# Run once if Qiskit is not installed in this Jupyter environment.
# %pip install "qiskit>=2.0,<3" matplotlib pylatexenc

from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Any, Mapping, Sequence

from qiskit import QuantumCircuit
from qiskit.circuit import Gate
from qiskit.circuit.library import UnitaryGate
from qiskit.exceptions import QiskitError
from qiskit.quantum_info import Operator


## Result data structures

`slot = i` means “immediately before backbone instruction `i`”; `slot = len(backbone)` is the position after the final backbone instruction. `source_indices` records which original instructions were combined into each backbone instruction.

In [ ]:
@dataclass(frozen=True)
class StoredOneQGate:
    operation: Gate
    logical_qubit: int
    slot: int
    source_index: int


@dataclass(frozen=True)
class BackboneInstruction:
    operation: Any
    qargs: tuple[int, ...]
    cargs: tuple[int, ...]
    source_indices: tuple[int, ...]


@dataclass(frozen=True)
class PreprocessingResult:
    backbone: QuantumCircuit
    one_qubit_gates: tuple[StoredOneQGate, ...]
    instructions: tuple[BackboneInstruction, ...]


## Preprocessor

The small temporary circuit in `_fuse_two_qubit_gates` is intentional. It lets Qiskit handle its own matrix/qubit-order convention, including the case where the second gate lists the same pair in reverse order. Parameterized gates can be fused once their parameters are bound; if conversion to a numeric unitary fails, the pair is simply left unfused.

In [ ]:
def _is_unitary_gate_of_width(operation: Any, width: int) -> bool:
    """Return True only for Gate objects of the requested width."""
    return isinstance(operation, Gate) and operation.num_qubits == width


def _fuse_two_qubit_gates(
    left: BackboneInstruction,
    right_operation: Gate,
    right_qargs: tuple[int, int],
) -> UnitaryGate:
    """Return the unitary for `left` followed by `right_operation`.

    The returned gate uses `left.qargs` as its operand order.
    """
    local = QuantumCircuit(2)
    local.append(left.operation, [0, 1])
    right_local_qargs = [left.qargs.index(q) for q in right_qargs]
    local.append(right_operation, right_local_qargs)

    names = [getattr(left.operation, "name", "gate"), right_operation.name]
    label = "fused(" + ",".join(names) + ")"
    return UnitaryGate(Operator(local).data, label=label)


def preprocess_for_swap_routing(circuit: QuantumCircuit) -> PreprocessingResult:
    """Extract 1q gates and fuse truly adjacent 2q gates on the same pair.

    Only unitary one-qubit Gate objects are extracted. Non-unitary or
    scheduling instructions such as measure/reset/delay are retained.
    """
    entries: list[BackboneInstruction] = []
    stored: list[StoredOneQGate] = []
    previous_input_was_two_qubit_gate = False

    for source_index, instruction in enumerate(circuit.data):
        operation = instruction.operation
        qargs = tuple(circuit.find_bit(q).index for q in instruction.qubits)
        cargs = tuple(circuit.find_bit(c).index for c in instruction.clbits)

        if _is_unitary_gate_of_width(operation, 1) and not cargs:
            stored.append(
                StoredOneQGate(
                    operation=operation,
                    logical_qubit=qargs[0],
                    slot=len(entries),
                    source_index=source_index,
                )
            )
            # A removed 1q gate still breaks adjacency in the original stream.
            previous_input_was_two_qubit_gate = False
            continue

        is_two_qubit_gate = (
            _is_unitary_gate_of_width(operation, 2) and not cargs
        )
        same_pair_as_previous = (
            previous_input_was_two_qubit_gate
            and bool(entries)
            and len(entries[-1].qargs) == 2
            and set(entries[-1].qargs) == set(qargs)
        )

        if is_two_qubit_gate and same_pair_as_previous:
            try:
                fused = _fuse_two_qubit_gates(entries[-1], operation, qargs)
            except (QiskitError, TypeError, ValueError):
                # Common case: an unbound Parameter prevents a numeric matrix.
                entries.append(
                    BackboneInstruction(operation, qargs, cargs, (source_index,))
                )
            else:
                entries[-1] = replace(
                    entries[-1],
                    operation=fused,
                    source_indices=entries[-1].source_indices + (source_index,),
                )
        else:
            entries.append(
                BackboneInstruction(operation, qargs, cargs, (source_index,))
            )

        previous_input_was_two_qubit_gate = is_two_qubit_gate

    backbone = circuit.copy_empty_like(name=f"{circuit.name}_2q_backbone")
    for entry in entries:
        backbone.append(entry.operation, list(entry.qargs), list(entry.cargs))

    return PreprocessingResult(
        backbone=backbone,
        one_qubit_gates=tuple(stored),
        instructions=tuple(entries),
    )

## Restoration helpers

`restore_without_routing` is useful for validation and reconstructs the preprocessed ordering. For a SWAP-inserting agent, call `mapped_one_qubit_gates_for_slot` with the agent's current mapping **before** it emits the backbone instruction for that slot.

The mapping convention below is `logical_to_physical[logical_qubit] -> physical_qubit`.

In [ ]:
def restore_without_routing(result: PreprocessingResult) -> QuantumCircuit:
    """Reinsert stored 1q gates when no logical-to-physical remapping occurred."""
    restored = result.backbone.copy_empty_like(name="restored_preprocessed")
    gates_by_slot: dict[int, list[StoredOneQGate]] = {}
    for stored_gate in result.one_qubit_gates:
        gates_by_slot.setdefault(stored_gate.slot, []).append(stored_gate)

    for slot in range(len(result.instructions) + 1):
        for stored_gate in gates_by_slot.get(slot, []):
            restored.append(stored_gate.operation, [stored_gate.logical_qubit])
        if slot < len(result.instructions):
            entry = result.instructions[slot]
            restored.append(entry.operation, list(entry.qargs), list(entry.cargs))
    return restored


def mapped_one_qubit_gates_for_slot(
    result: PreprocessingResult,
    slot: int,
    logical_to_physical: Mapping[int, int] | Sequence[int],
) -> list[tuple[Gate, int]]:
    """Translate stored 1q gates at one backbone boundary to physical qubits."""
    return [
        (stored_gate.operation, logical_to_physical[stored_gate.logical_qubit])
        for stored_gate in result.one_qubit_gates
        if stored_gate.slot == slot
    ]

## Example and correctness checks

In [ ]:
qc = QuantumCircuit(3, name="example")
qc.h(0)                 # extracted: slot 0
qc.cx(0, 1)
qc.cz(1, 0)             # fused with CX despite reversed qarg order
qc.rz(0.3, 2)           # extracted: slot 1; breaks adjacency
qc.cx(1, 2)
qc.x(1)                 # extracted: slot 2; breaks adjacency
qc.cx(1, 2)             # therefore not fused with the previous CX

result = preprocess_for_swap_routing(qc)
restored = restore_without_routing(result)

print("Original circuit:")
display(qc.draw("mpl"))
print("Two-qubit backbone:")
display(result.backbone.draw("mpl"))
print("Stored one-qubit gates:")
for item in result.one_qubit_gates:
    print(item)

assert len(result.one_qubit_gates) == 3
assert len(result.instructions) == 3
assert result.instructions[0].source_indices == (1, 2)
assert Operator(qc).equiv(Operator(restored))
print("\nPassed: the restored circuit is unitary-equivalent to the input.")

## How the RL router should consume the result

For each `slot` from `0` through `len(result.instructions)`:

1. Emit the stored one-qubit gates returned by `mapped_one_qubit_gates_for_slot(...)` using the router's current logical-to-physical mapping.
2. If this is not the final slot, insert any SWAPs chosen by the agent and update the mapping.
3. Emit `result.instructions[slot]`, mapping each logical operand in its `qargs` through that same current mapping.

Whether step 1 occurs before or after newly selected SWAPs is a routing-policy choice. To reproduce the original logical semantics, use the mapping that is active at the exact point where the stored gate is emitted. Keep this convention consistent when generating the agent's output circuit.